# Solving Odd Stokes Surface Flow on a Genus n Surface

We want to solve
$$
\begin{alignat}{3}
- \mathbf{P} div_{\Gamma} (E(\mathbf{u}_T)) + \nabla_{\Gamma} p + \alpha \mathbf{u} &= \mathbf{f}_T && \quad \text{on} \Gamma
\\
div_{\Gamma} \mathbf{u}_T &= g && \text{on} \Gamma.
\end{alignat}
$$


In [15]:
import numpy as np
from netgen.csg import *
from netgen.meshing import MeshingStep
from ngsolve import *
from ngsolve.webgui import Draw
ngsglobals.msg_level = 1

In [17]:
sys.path.insert(0, "../../sandbox/")
from generate_surface_meshes import *

maxh = 0.1
mesh = generate_n_torus_mesh(maxh, order_g = 5, char_len=1, n=4)

Draw(mesh)

 Curve elements, order = 5


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [18]:
# Define geometry related objects and differential operators
# Mesh size
h = specialcf.mesh_size

# Normal vector
ns = specialcf.normal(mesh.dim)
# normal and tangential  projection
Qs = OuterProduct(ns, ns)
Ps = Id(mesh.dim) - Qs
# Weingarten map
Bs = Grad(ns)
# TODO: Or should we rather use
# specialcf.Weingarten(mesh.dim)

# Define covariant derivative for vectors
def Nabla_h(u):
    return Ps*Grad(u).Trace()*Ps
# and its corrected version for non-tangential vectors
def Nabla_th(u):
    return Nabla_(u) - InnerProduct(u, ns)*Bs

# Define discrete strain rate tensor
# TODO: Need to clarify meaning/implementation of Grad(u).Trace()
#       for VectorH1
# TODO: For now we safeguard with Projection from both sides
def E_h(u):
    return Ps*Sym(Grad(u).Trace())*Ps

# Corrected strain rate tensor for non-tangential 
def E_th(u):
    return E_h(u) - InnerProduct(u, ns)*Bs

Compute mean curvature, its gradient and resulting forces.
*TODO:*
Improve both meshing of implicit surfaces and curvature related computations.

In [19]:
H = Trace(Bs)
Draw(H,mesh)
VH=H1(mesh,order=4)
H_grid=GridFunction(VH)
H_grid.Set(H,definedon=mesh.Boundaries(".*"))
Draw(H_grid)
gradH=Cross(Ps*Grad(H_grid),ns)
Draw(gradH,mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

Next, we define the discrete weak formulation of surface Stokes problem with
driving forces given by 90 degree in surface rotation of the surface gradient of mean curvature.

In [20]:
order_k = 2

V = VectorH1(mesh, order=int(order_k))
Q = H1(mesh, order=int(order_k-1))
N = NumberSpace(mesh)
W = V*Q*N
(u, p, lam), (v, q, mu) = W.TnT()

# Penalty parameter to weakly enforce tangential condition
beta = 1.0

Astar = BilinearForm(W)
Astar += 2*InnerProduct(E_th(u), E_th(v))*ds
Astar += InnerProduct(Ps*u.Trace(), Ps*v.Trace())*ds
Astar += beta*h**(-2.0)*InnerProduct(Qs*u.Trace(), Qs*v.Trace())*ds
Astar += InnerProduct(grad(p).Trace(), v.Trace())*ds
Astar += InnerProduct(grad(q).Trace(), u.Trace())*ds
Astar += (p*mu + q*lam)*ds
Astar.Assemble()

# Odd mechanics-like force field 
f = gradH
l = LinearForm(W)
l += InnerProduct(f, v)*ds
l.Assemble()

Finally,  we solve problem and write out solution.

*TODO:* 
At the moment the generating mesh is not entirely smooth
and the mean curvature approximations are crude resulting in some
local, artificial spikes in the flow.

In [21]:
up_h =  GridFunction(W)
Ainv = Astar.mat.Inverse(freedofs=W.FreeDofs())
up_h.vec.data = Ainv*l.vec

u_h, p_h, _ = up_h.components

Draw(u_h)
Draw(p_h)
Draw(f,mesh)
vtkfile = VTKOutput(mesh, coefs=[u_h, f, p_h], names=["u_h", "f", "p_h"], filename="results_stokes",subdivision=3)
vtkfile.Do(vb=BND)
Draw(mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene